In [1]:
import os
import numpy as np
import pandas as pd
import skimage
from tqdm import tqdm
import subprocess
from tqdm import tqdm

In [4]:
folders = ['train', 'validation', 'test']

for folder in folders:
    LOAD_PATH = f'/scr/yren/annotated_mn_datasets/{folder}/'
    SAVE_PATH = f'/scr/yren/microsam_data/{folder}/'
    
    images = os.listdir(os.path.join(LOAD_PATH, 'images'))
    images = [image for image in images if not image.startswith('.')]
    images.sort()
    
    gts = os.listdir(os.path.join(LOAD_PATH, 'mn_masks'))
    gts = [gt for gt in gts if not gt.startswith('.')]
    gts.sort()
    
    PS = 256
    for i in tqdm(range(len(gts))):
        im = skimage.io.imread(os.path.join(LOAD_PATH, 'images', images[i]))
        gt = skimage.io.imread(os.path.join(LOAD_PATH, 'mn_masks', gts[i]))

        assert im.shape == gt.shape
        H,W = im.shape
        patches_per_image = (W // PS) * (H // PS)
        X = np.linspace(0, W - W % PS, W // PS + 1)
        Y = np.linspace(0, H - H % PS, H // PS + 1)
        X,Y = np.meshgrid(X[:-1],Y[:-1], indexing='ij')
        X = X.reshape((patches_per_image,))
        Y = Y.reshape((patches_per_image,))
        C = np.stack((Y,X)).T
        
        idx = 0
        for j in range(len(C)):
            r,c = C[j]
            r,c = int(r), int(c)
            # check if gt patch has micronuclei or not
            gt_patch = gt[r:r+PS, c:c+PS]
            gt_labels = skimage.morphology.label(gt_patch)
            if np.max(gt_labels) > 0: # 1 represents only background
                # save the corresponding input and gt
                im_patch = im[r:r+PS, c:c+PS]
                
                # reconstruct filename
                imid = images[i].split('.')[0]
                suffix = f'.crop{idx}'
                new_imid = imid + suffix
                
                # im_patch = (im_patch - np.min(im_patch)) / (np.max(im_patch) - np.min(im_patch)) * 255
                skimage.io.imsave(os.path.join(SAVE_PATH, 'images', new_imid + '.tif'), im_patch)
                skimage.io.imsave(os.path.join(SAVE_PATH, 'mn_masks', new_imid + '.png'), gt_labels.astype(np.uint16))
                idx = idx + 1

  0%|          | 0/121 [00:00<?, ?it/s]/tmp/ipykernel_1355371/3194372754.py:48: UserWarning: /scr/yren/microsam_data/train/mn_masks/2022-03-18_RPE1-3.crop0.png is a low contrast image
  skimage.io.imsave(os.path.join(SAVE_PATH, 'mn_masks', new_imid + '.png'), gt_labels.astype(np.uint16))
/tmp/ipykernel_1355371/3194372754.py:48: UserWarning: /scr/yren/microsam_data/train/mn_masks/2022-03-18_RPE1-3.crop1.png is a low contrast image
  skimage.io.imsave(os.path.join(SAVE_PATH, 'mn_masks', new_imid + '.png'), gt_labels.astype(np.uint16))
/tmp/ipykernel_1355371/3194372754.py:48: UserWarning: /scr/yren/microsam_data/train/mn_masks/2022-03-18_RPE1-3.crop2.png is a low contrast image
  skimage.io.imsave(os.path.join(SAVE_PATH, 'mn_masks', new_imid + '.png'), gt_labels.astype(np.uint16))
/tmp/ipykernel_1355371/3194372754.py:48: UserWarning: /scr/yren/microsam_data/train/mn_masks/2022-03-18_RPE1-3.crop3.png is a low contrast image
  skimage.io.imsave(os.path.join(SAVE_PATH, 'mn_masks', new_imid +